#### DATA QUALITY DRIFT DETECTION

%md

##### 17.1 DOCUMENTAÇÃO DO DATA QUALITY DRIFT DETECTION

###### Objetivo

O **Data Quality Drift Detection** compara duas avaliações previamente calculadas do mesmo dataset: uma referência (**baseline**) e uma avaliação **atual**. O módulo identifica mudanças estruturais e variações em indicadores numéricos de qualidade, sem reexecutar Profiles nem reler os registros originais.

A análise busca responder:

> Quais métricas e atributos mudaram entre as duas avaliações, quais mudanças excedem os limites configurados e quais resultados são realmente comparáveis?

O módulo tem caráter **comparativo e exploratório**. Uma mudança estatística não é, por si só, um erro, nem comprova degradação.

---

###### Exemplos

**Exemplo 1 — Queda de completude**

Uma métrica `pct_preenchimento` passa de 98% para 81%. O módulo registra a diferença de **−17 pontos percentuais**. A classificação de regressão só ocorre se a métrica tiver direção explícita `HIGHER_BETTER`.

**Exemplo 2 — Aumento de violações**

Uma taxa de violações passa de 1% para 4%. Com direção `LOWER_BETTER`, a mudança é interpretada como possível regressão.

**Exemplo 3 — Coluna adicionada**

O snapshot atual contém um atributo ausente na referência. A mudança é registrada como `ADICIONADO`, sem presumir problema de qualidade.

**Exemplo 4 — Mudança de tipo**

Uma coluna muda de `integer` para `string`. O módulo registra um evento de `TIPO_ALTERADO` quando ambos os snapshots disponibilizam seus metadados estruturais.

**Exemplo 5 — Métrica sem comparabilidade**

Uma métrica existe apenas na referência ou apresenta unidade diferente. O módulo registra a situação, mas não calcula um delta enganoso.

---

###### Principais análises

1. **Configuração:** fontes Delta ou DataFrames em memória, identificadores e limiares.
2. **Contratos:** verificação de campos, chaves e tipos numéricos.
3. **Normalização:** transformação de métricas agregadas em contrato canônico.
4. **Comparabilidade:** verificação de identidade, unidade e versão dos critérios.
5. **Schema Drift:** atributos adicionados, removidos e mudanças de tipo.
6. **Metric Drift:** diferenças absolutas e relativas.
7. **Classificação:** mudanças relevantes e regressões conforme política explícita.
8. **Cobertura:** inventário de métricas comparáveis e indisponíveis.
9. **Resumo:** contagens e limitações da comparação.
10. **Testes:** verificações sintéticas opcionais.

---

###### Métodos de execução

**Entrada agregada:** os snapshots utilizam métricas pré-calculadas, não linhas do dataset original.

**Modos `MEMORY` e `DELTA`:** no modo `MEMORY`, o usuário fornece DataFrames Spark já disponíveis nesta sessão. No modo `DELTA`, informa nomes de tabelas existentes. Nenhuma fonte é descoberta por suposição.

**Contrato canônico:** `dimensao`, `atributo`, `metrica`, `valor`, `unidade`, `versao_criterio`. A chave de comparação é a combinação de dimensão, atributo e métrica. Valores ausentes ou unidades divergentes não geram delta.

**Direção de qualidade:** somente métricas explicitamente cadastradas em `DRIFT_METRIC_DIRECTIONS` podem ser classificadas como melhora ou regressão. As demais recebem interpretação neutra.

**Tolerância:** diferenças absolutas e relativas são parâmetros operacionais. Para métricas percentuais, o delta absoluto é expresso em pontos percentuais.

---

###### Tratamento de situações especiais

- **Fonte inexistente:** interrompe a execução quando a análise estiver habilitada;
- **Execução desabilitada:** não acessa tabelas nem dispara jobs de comparação;
- **Dataset diferente:** bloqueia a comparação;
- **Versão de critérios diferente:** métricas são marcadas como incompatíveis;
- **Unidade diferente:** não calcula delta;
- **Valor nulo ou não finito:** não calcula delta;
- **Denominador zero:** delta relativo permanece nulo;
- **Métrica ausente em um snapshot:** registra `SOMENTE_REFERENCIA` ou `SOMENTE_ATUAL`;
- **Chaves duplicadas:** rejeita o snapshot para evitar multiplicação em joins;
- **Schema indisponível:** registra a ausência de avaliação estrutural;
- **Atributos novos ou removidos:** sinaliza mudança, sem afirmar regressão;
- **Mudança de tipo:** registra evento estrutural para investigação.

---

###### Interpretação dos resultados

**`delta_absoluto`:** valor atual menos valor de referência, na unidade original.

**`delta_relativo_pct`:** variação percentual em relação ao módulo da referência; indefinida para referência zero.

**`mudanca_relevante`:** ultrapassagem de limiar configurado, sem pressupor erro.

**`interpretacao`:** `REGRESSAO`, `MELHORA`, `ALTERACAO_NEUTRA`, `ESTAVEL` ou `NAO_COMPARAVEL`.

**Princípio fundamental:**

> Drift é mudança observada; regressão de qualidade exige uma política explícita sobre a direção e o significado da métrica.

---

###### Características do módulo

- **Genérico:** sem regras específicas do setor elétrico ou de qualquer domínio;
- **Modular:** não executa Profiles nem altera o Score;
- **Rastreável:** preserva IDs de referência e atual;
- **Conservador:** não compara valores com unidades ou critérios incompatíveis;
- **Escalável:** opera sobre resultados agregados em Spark;
- **Não destrutivo:** não grava tabelas por padrão;
- **Configurável:** limites, direção e fontes explícitos.

---

###### DataFrames de resultado

**`drift_sources_df`:** fontes e metadados dos snapshots.

**`drift_comparability_df`:** métricas presentes, ausentes ou incompatíveis.

**`drift_metrics_df`:** diferenças, limiares e interpretações.

**`drift_schema_df`:** mudanças de atributos e tipos, quando fornecidos.

**`drift_summary_df`:** resumo executivo e cobertura da comparação.

---

###### Limitações

1. **Requer dois snapshots:** sem referência e avaliação atual não existe drift.
2. **Não cria histórico:** snapshots devem ser disponibilizados externamente.
3. **Sem testes de distribuição completa:** PSI, KS ou Jensen–Shannon exigem histogramas ou frequências compatíveis.
4. **Sem causalidade:** mudanças não comprovam a causa de um problema.
5. **Sem correção automática:** nenhuma alteração é aplicada aos dados.
6. **Sem aprovação automática:** o resultado não altera o Data Quality Score.
7. **Sensibilidade aos limiares:** a classificação depende dos parâmetros escolhidos.
8. **Metadados opcionais:** sem snapshots estruturais, Schema Drift não é avaliado.

---

###### Resultado esperado

Ao final, o módulo deverá permitir responder:

1. Quais métricas são comparáveis?
2. Quais métricas estão ausentes em cada snapshot?
3. Quais métricas tiveram variação relevante?
4. Quais mudanças representam regressão segundo critérios explícitos?
5. Quais atributos foram adicionados, removidos ou tiveram tipo alterado?
6. Qual é a cobertura efetiva da comparação?
7. Quais limitações impedem conclusões mais amplas?

**Observação:** o módulo não interpreta toda mudança como degradação e não utiliza informações do dataset original.


In [0]:
%run "../bibliotecas/00_FRAMEWORK_CONFIG"

In [0]:
# ============================================================
# 17.3 CONFIGURAÇÕES E FONTES
# ============================================================

# O QUE FAZ:
# Define snapshots, contratos e parâmetros de comparação.

# COMO FAZ:
# Utiliza referências explícitas a DataFrames ou tabelas Delta.

# POR QUE É IMPORTANTE:
# Evita reprocessamento e mistura de execuções.

# PERGUNTA RESPONDIDA:
# "Quais avaliações serão comparadas?"


DRIFT_ENABLE = False
DRIFT_RUN_TESTS = False
DRIFT_SOURCE_MODE = "MEMORY"  # MEMORY ou DELTA

# MEMORY: atribua DataFrames Spark já calculados, nesta sessão.
DRIFT_BASELINE_DF = None
DRIFT_CURRENT_DF = None
DRIFT_BASELINE_SCHEMA_DF = None
DRIFT_CURRENT_SCHEMA_DF = None

# DELTA: indique nomes reais de tabelas previamente persistidas.
DRIFT_BASELINE_TABLE = None
DRIFT_CURRENT_TABLE = None
DRIFT_BASELINE_SCHEMA_TABLE = None
DRIFT_CURRENT_SCHEMA_TABLE = None

DRIFT_DATASET_ID = None            # Obrigatório ao habilitar
DRIFT_BASELINE_RUN_ID = None       # Obrigatório ao habilitar
DRIFT_CURRENT_RUN_ID = None        # Obrigatório ao habilitar

# Valores de 'valor' devem ser numéricos, em unidade explicitamente identificada.
DRIFT_ABSOLUTE_THRESHOLD = 5.0
DRIFT_RELATIVE_THRESHOLD_PCT = 20.0
DRIFT_MAX_METRICS = 100000        # Limite de proteção para snapshots agregados
DRIFT_DISPLAY_RESULTS = True

# Direção explícita: HIGHER_BETTER, LOWER_BETTER ou NEUTRAL.
# A ausência de uma métrica no catálogo significa NEUTRAL.
DRIFT_METRIC_DIRECTIONS = {
    "pct_preenchimento": "HIGHER_BETTER",
    "pct_completude_efetiva": "HIGHER_BETTER",
    "score_geral": "HIGHER_BETTER",
    "pct_violacoes": "LOWER_BETTER",
}

In [0]:
# ============================================================
# 17.4 CONTRATOS E VALIDAÇÃO DOS PARÂMETROS
# ============================================================

# O QUE FAZ:
# Valida parâmetros e estabelece os contratos canônicos.

# COMO FAZ:
# Confere tipos, limiares, IDs e funções compartilhadas do 00.

# POR QUE É IMPORTANTE:
# Impede comparações sem referência ou com configurações ambíguas.

# PERGUNTA RESPONDIDA:
# "Os parâmetros permitem uma comparação válida?"


DRIFT_METRIC_COLUMNS = [
    "dimensao", "atributo", "metrica", "valor", "unidade", "versao_criterio"
]
DRIFT_SCHEMA_COLUMNS = ["atributo", "tipo"]
DRIFT_ALLOWED_DIRECTIONS = {"HIGHER_BETTER", "LOWER_BETTER", "NEUTRAL"}
if not 0 <= DRIFT_ABSOLUTE_THRESHOLD:
    raise ValueError("DRIFT_ABSOLUTE_THRESHOLD deve ser >= 0.")
if not 0 <= DRIFT_RELATIVE_THRESHOLD_PCT:
    raise ValueError("DRIFT_RELATIVE_THRESHOLD_PCT deve ser >= 0.")
if not isinstance(DRIFT_MAX_METRICS, int) or DRIFT_MAX_METRICS < 1:
    raise ValueError("DRIFT_MAX_METRICS deve ser inteiro positivo.")
if DRIFT_SOURCE_MODE not in {"MEMORY", "DELTA"}:
    raise ValueError("DRIFT_SOURCE_MODE deve ser MEMORY ou DELTA.")
if any(v not in DRIFT_ALLOWED_DIRECTIONS for v in DRIFT_METRIC_DIRECTIONS.values()):
    raise ValueError("DRIFT_METRIC_DIRECTIONS contém direção inválida.")
if "dq_col" not in globals() or "F" not in globals() or "DataFrame" not in globals():
    raise RuntimeError("O notebook 00 não disponibilizou dq_col, F e DataFrame.")
if DRIFT_ENABLE:
    if not all(isinstance(x, str) and x.strip() for x in (
        DRIFT_DATASET_ID, DRIFT_BASELINE_RUN_ID, DRIFT_CURRENT_RUN_ID
    )):
        raise ValueError("Defina dataset e run_ids de referência e atual.")
    if DRIFT_BASELINE_RUN_ID == DRIFT_CURRENT_RUN_ID:
        raise ValueError("Baseline e atual precisam ter run_ids distintos.")
print("Drift habilitado:", DRIFT_ENABLE)

In [0]:
# ============================================================
# 17.5 LEITURA CONTROLADA DOS SNAPSHOTS
# ============================================================

# O QUE FAZ:
# Obtém somente os resultados agregados explicitamente configurados.

# COMO FAZ:
# Lê DataFrames em memória ou tabelas Delta, sem chamar Profiles.

# POR QUE É IMPORTANTE:
# Evita custos inesperados e dependência de sessões isoladas.

# PERGUNTA RESPONDIDA:
# "Os dois snapshots estão acessíveis?"


def drift_read_source(memory_df, table_name, required, label, optional=False):
    if DRIFT_SOURCE_MODE == "MEMORY":
        source = memory_df
    else:
        source = spark.table(table_name) if table_name else None
    if source is None:
        if optional:
            return None
        raise ValueError("Fonte obrigatória ausente: " + label)
    if not isinstance(source, DataFrame):
        raise TypeError("Fonte " + label + " deve ser DataFrame Spark.")
    missing = [c for c in required if c not in source.columns]
    if missing:
        raise ValueError("Campos ausentes em " + label + ": " + str(missing))
    return source

drift_baseline_df = None
drift_current_df = None
drift_baseline_schema_df = None
drift_current_schema_df = None
if DRIFT_ENABLE:
    drift_baseline_df = drift_read_source(
        DRIFT_BASELINE_DF, DRIFT_BASELINE_TABLE, DRIFT_METRIC_COLUMNS, "baseline"
    )
    drift_current_df = drift_read_source(
        DRIFT_CURRENT_DF, DRIFT_CURRENT_TABLE, DRIFT_METRIC_COLUMNS, "atual"
    )
    drift_baseline_schema_df = drift_read_source(
        DRIFT_BASELINE_SCHEMA_DF, DRIFT_BASELINE_SCHEMA_TABLE,
        DRIFT_SCHEMA_COLUMNS, "schema baseline", optional=True
    )
    drift_current_schema_df = drift_read_source(
        DRIFT_CURRENT_SCHEMA_DF, DRIFT_CURRENT_SCHEMA_TABLE,
        DRIFT_SCHEMA_COLUMNS, "schema atual", optional=True
    )

In [0]:
# ============================================================
# 17.6 NORMALIZAÇÃO E INTEGRIDADE
# ============================================================

# O QUE FAZ:
# Normaliza métricas e valida unicidade das chaves.

# COMO FAZ:
# Converte valores para double e rejeita duplicatas ou volumes inesperados.

# POR QUE É IMPORTANTE:
# Evita joins multiplicativos e métricas ambíguas.

# PERGUNTA RESPONDIDA:
# "Os snapshots podem ser comparados com segurança?"


def drift_normalize_metrics(source, label):
    df = source.select(
        *[dq_col(c).cast("string").alias(c) for c in
          ("dimensao", "atributo", "metrica", "unidade", "versao_criterio")],
        dq_col("valor").cast("double").alias("valor")
    )
    if df.filter(
        F.col("dimensao").isNull() | F.col("metrica").isNull()
        | F.col("unidade").isNull() | F.col("versao_criterio").isNull()
        | F.col("atributo").isNull()
    ).limit(1).count():
        raise ValueError("Chaves/metadados nulos em " + label)
    if df.limit(DRIFT_MAX_METRICS + 1).count() > DRIFT_MAX_METRICS:
        raise ValueError("Snapshot " + label + " excede DRIFT_MAX_METRICS.")
    keys = ["dimensao", "atributo", "metrica"]
    if df.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Chaves duplicadas em " + label)
    return df

drift_baseline_norm_df = None
drift_current_norm_df = None
if DRIFT_ENABLE:
    drift_baseline_norm_df = drift_normalize_metrics(drift_baseline_df, "baseline")
    drift_current_norm_df = drift_normalize_metrics(drift_current_df, "atual")

In [0]:
# ============================================================
# 17.7 COMPARABILIDADE DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Classifica métricas comparáveis, ausentes ou incompatíveis.

# COMO FAZ:
# Realiza full outer join por chave e compara unidades e critérios.

# POR QUE É IMPORTANTE:
# Impede deltas matematicamente válidos, mas semanticamente incorretos.

# PERGUNTA RESPONDIDA:
# "Quais métricas podem ser comparadas?"


comparability_schema = (
    "dimensao string, atributo string, metrica string, "
    "valor_referencia double, valor_atual double, "
    "unidade_referencia string, unidade_atual string, "
    "versao_referencia string, versao_atual string, status_comparabilidade string"
)
if DRIFT_ENABLE:
    keys = ["dimensao", "atributo", "metrica"]
    b = drift_baseline_norm_df.withColumn("_presente_b", F.lit(True)).alias("b")
    a = drift_current_norm_df.withColumn("_presente_a", F.lit(True)).alias("a")
    joined = b.join(a, keys, "full_outer")
    drift_comparability_df = joined.select(
        *[F.col(k) for k in keys],
        F.col("b.valor").alias("valor_referencia"),
        F.col("a.valor").alias("valor_atual"),
        F.col("b.unidade").alias("unidade_referencia"),
        F.col("a.unidade").alias("unidade_atual"),
        F.col("b.versao_criterio").alias("versao_referencia"),
        F.col("a.versao_criterio").alias("versao_atual"),
        F.when(F.col("_presente_b").isNull(), "SOMENTE_ATUAL")
         .when(F.col("_presente_a").isNull(), "SOMENTE_REFERENCIA")
         .when(~F.col("b.unidade").eqNullSafe(F.col("a.unidade")), "UNIDADE_DIFERENTE")
         .when(~F.col("b.versao_criterio").eqNullSafe(F.col("a.versao_criterio")), "CRITERIO_DIFERENTE")
         .when(
             F.col("b.valor").isNull() | F.col("a.valor").isNull()
             | F.isnan(F.col("b.valor")) | F.isnan(F.col("a.valor"))
             | (F.abs(F.col("b.valor")) == float("inf"))
             | (F.abs(F.col("a.valor")) == float("inf")),
             "VALOR_INVALIDO"
         )
         .otherwise("COMPARAVEL").alias("status_comparabilidade")
    )
else:
    drift_comparability_df = spark.createDataFrame([], comparability_schema)

In [0]:
# ============================================================
# 17.8 DETECÇÃO DE METRIC DRIFT
# ============================================================

# O QUE FAZ:
# Calcula diferenças absolutas e relativas de métricas comparáveis.

# COMO FAZ:
# Aplica expressões Spark sem coletar valores individuais ao driver.

# POR QUE É IMPORTANTE:
# Permite medir mudanças mantendo unidades e critérios.

# PERGUNTA RESPONDIDA:
# "Quais indicadores apresentaram variações relevantes?"


metrics_schema = (
    comparability_schema + ", delta_absoluto double, delta_relativo_pct double, "
    "mudanca_relevante boolean, direcao_qualidade string, interpretacao string"
)
if DRIFT_ENABLE:
    # Expressão literal pequena: políticas são configurações, não dados de origem.
    direction_expr = F.create_map(*[
        x for key, value in DRIFT_METRIC_DIRECTIONS.items()
        for x in (F.lit(key), F.lit(value))
    ]) if DRIFT_METRIC_DIRECTIONS else None
    d = drift_comparability_df
    comparable = F.col("status_comparabilidade") == "COMPARAVEL"
    delta = F.col("valor_atual") - F.col("valor_referencia")
    relative = F.when(
        F.col("valor_referencia") != 0,
        delta / F.abs(F.col("valor_referencia")) * F.lit(100.0)
    ).otherwise(F.lit(None).cast("double"))
    direction = (
        F.coalesce(F.element_at(direction_expr, F.col("metrica")), F.lit("NEUTRAL"))
        if direction_expr is not None else F.lit("NEUTRAL")
    )
    drift_metrics_df = (
        d.withColumn("delta_absoluto", F.when(comparable, delta))
         .withColumn("delta_relativo_pct", F.when(comparable, relative))
         .withColumn("direcao_qualidade", direction)
         .withColumn(
             "mudanca_relevante",
             F.when(~comparable, F.lit(None).cast("boolean"))
              .otherwise(
                  (F.abs(F.col("delta_absoluto")) > DRIFT_ABSOLUTE_THRESHOLD)
                  | (F.abs(F.col("delta_relativo_pct")) > DRIFT_RELATIVE_THRESHOLD_PCT)
              )
         )
         .withColumn(
             "interpretacao",
             F.when(~comparable, "NAO_COMPARAVEL")
              .when(~F.col("mudanca_relevante"), "ESTAVEL")
              .when(F.col("direcao_qualidade") == "NEUTRAL", "ALTERACAO_NEUTRA")
              .when(
                  ((F.col("direcao_qualidade") == "HIGHER_BETTER") & (F.col("delta_absoluto") < 0))
                  | ((F.col("direcao_qualidade") == "LOWER_BETTER") & (F.col("delta_absoluto") > 0)),
                  "REGRESSAO"
              ).otherwise("MELHORA")
         )
    )
else:
    drift_metrics_df = spark.createDataFrame([], metrics_schema)

In [0]:
# ============================================================
# 17.9 DETECÇÃO DE SCHEMA DRIFT
# ============================================================

# O QUE FAZ:
# Identifica atributos adicionados, removidos e alterações de tipo.

# COMO FAZ:
# Compara snapshots estruturais opcionais por nome de atributo.

# POR QUE É IMPORTANTE:
# Evidencia mudanças de contrato sem reler o dataset.

# PERGUNTA RESPONDIDA:
# "A estrutura dos atributos mudou?"


schema_events_schema = (
    "atributo string, tipo_referencia string, tipo_atual string, evento string"
)
drift_schema_status = "NAO_AVALIADO"
if DRIFT_ENABLE and drift_baseline_schema_df is not None and drift_current_schema_df is not None:
    def drift_normalize_schema(source, label):
        df = source.select(
            dq_col("atributo").cast("string").alias("atributo"),
            dq_col("tipo").cast("string").alias("tipo")
        )
        if df.filter(F.col("atributo").isNull() | F.col("tipo").isNull()).limit(1).count():
            raise ValueError("Schema contém atributos/tipos nulos: " + label)
        if df.groupBy("atributo").count().filter(F.col("count") > 1).limit(1).count():
            raise ValueError("Schema contém atributos duplicados: " + label)
        return df
    b = drift_normalize_schema(drift_baseline_schema_df, "baseline").alias("b")
    a = drift_normalize_schema(drift_current_schema_df, "atual").alias("a")
    drift_schema_df = (
        b.join(a, "atributo", "full_outer")
        .select(
            "atributo",
            F.col("b.tipo").alias("tipo_referencia"),
            F.col("a.tipo").alias("tipo_atual")
        )
        .withColumn(
            "evento",
            F.when(F.col("tipo_referencia").isNull(), "ADICIONADO")
             .when(F.col("tipo_atual").isNull(), "REMOVIDO")
             .when(F.col("tipo_referencia") != F.col("tipo_atual"), "TIPO_ALTERADO")
             .otherwise("SEM_ALTERACAO")
        )
    )
    drift_schema_status = "AVALIADO"
else:
    drift_schema_df = spark.createDataFrame([], schema_events_schema)

In [0]:
# ============================================================
# 17.10 INVENTÁRIO DAS FONTES
# ============================================================

# O QUE FAZ:
# Registra os snapshots e o estado da comparação.

# COMO FAZ:
# Cria metadados operacionais sem persistência.

# POR QUE É IMPORTANTE:
# Preserva a rastreabilidade das avaliações utilizadas.

# PERGUNTA RESPONDIDA:
# "Qual é a origem dos resultados comparados?"


drift_sources_df = spark.createDataFrame(
    [
        ("BASELINE", str(DRIFT_BASELINE_RUN_ID or ""), str(DRIFT_DATASET_ID or ""),
         DRIFT_SOURCE_MODE, bool(DRIFT_ENABLE)),
        ("ATUAL", str(DRIFT_CURRENT_RUN_ID or ""), str(DRIFT_DATASET_ID or ""),
         DRIFT_SOURCE_MODE, bool(DRIFT_ENABLE))
    ],
    "papel string, run_id string, dataset_id string, modo_leitura string, habilitado boolean"
)


In [0]:
# ============================================================
# 17.11 COBERTURA E CLASSIFICAÇÃO
# ============================================================

# O QUE FAZ:
# Consolida estados de comparabilidade e interpretações.

# COMO FAZ:
# Agrupa indicadores já calculados por situação.

# POR QUE É IMPORTANTE:
# Distingue cobertura insuficiente de estabilidade real.

# PERGUNTA RESPONDIDA:
# "Qual é a abrangência efetiva da análise?"


coverage_schema = "categoria string, classificacao string, quantidade long"
if DRIFT_ENABLE:
    metric_counts = drift_metrics_df.groupBy("interpretacao").count().select(
        F.lit("METRICA").alias("categoria"),
        F.col("interpretacao").alias("classificacao"),
        F.col("count").cast("long").alias("quantidade")
    )
    comparable_counts = drift_comparability_df.groupBy("status_comparabilidade").count().select(
        F.lit("COMPARABILIDADE").alias("categoria"),
        F.col("status_comparabilidade").alias("classificacao"),
        F.col("count").cast("long").alias("quantidade")
    )
    drift_coverage_df = metric_counts.unionByName(comparable_counts)
else:
    drift_coverage_df = spark.createDataFrame([], coverage_schema)

In [0]:
# ============================================================
# 17.12 RESUMO EXECUTIVO
# ============================================================

# O QUE FAZ:
# Consolida contagens, cobertura e estado do Schema Drift.

# COMO FAZ:
# Agrega DataFrames pequenos e calcula indicadores globais.

# POR QUE É IMPORTANTE:
# Permite interpretar resultados sem confundir ausência com estabilidade.

# PERGUNTA RESPONDIDA:
# "Quantas métricas mudaram e quantas foram comparáveis?"


summary_schema = (
    "status string, dataset_id string, run_referencia string, run_atual string, "
    "total_metricas long, metricas_comparaveis long, regressoes long, "
    "melhoras long, alteracoes_neutras long, pct_comparabilidade double, "
    "schema_status string, alteracoes_schema long"
)
if DRIFT_ENABLE:
    totals = drift_metrics_df.agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("status_comparabilidade") == "COMPARAVEL", 1).otherwise(0)).alias("comparaveis"),
        F.sum(F.when(F.col("interpretacao") == "REGRESSAO", 1).otherwise(0)).alias("regressoes"),
        F.sum(F.when(F.col("interpretacao") == "MELHORA", 1).otherwise(0)).alias("melhoras"),
        F.sum(F.when(F.col("interpretacao") == "ALTERACAO_NEUTRA", 1).otherwise(0)).alias("neutras")
    ).first()
    total = int(totals["total"] or 0)
    comparable = int(totals["comparaveis"] or 0)
    schema_changes = (
        drift_schema_df.filter(F.col("evento") != "SEM_ALTERACAO").count()
        if drift_schema_status == "AVALIADO" else 0
    )
    drift_summary_df = spark.createDataFrame([(
        "CONCLUIDO", DRIFT_DATASET_ID, DRIFT_BASELINE_RUN_ID, DRIFT_CURRENT_RUN_ID,
        total, comparable, int(totals["regressoes"] or 0),
        int(totals["melhoras"] or 0), int(totals["neutras"] or 0),
        float(100.0 * comparable / total) if total else None,
        drift_schema_status, int(schema_changes)
    )], summary_schema)
else:
    drift_summary_df = spark.createDataFrame([(
        "DESABILITADO", "", "", "", 0, 0, 0, 0, 0,
        None, "NAO_AVALIADO", 0
    )], summary_schema)

In [0]:
# ============================================================
# 17.13 TESTES SINTÉTICOS OPCIONAIS
# ============================================================

# O QUE FAZ:
# Verifica contratos e cálculo de diferenças sem usar dados reais.

# COMO FAZ:
# Cria dois snapshots pequenos com métricas e critérios conhecidos.

# POR QUE É IMPORTANTE:
# Detecta regressões sem reexecutar Profiles.

# PERGUNTA RESPONDIDA:
# "A lógica básica de comparação preserva os contratos?"


if DRIFT_RUN_TESTS:
    test_b = spark.createDataFrame(
        [("schema", "a", "pct_preenchimento", 95.0, "%", "v1")],
        "dimensao string, atributo string, metrica string, valor double, unidade string, versao_criterio string"
    )
    test_a = spark.createDataFrame(
        [("schema", "a", "pct_preenchimento", 80.0, "%", "v1")],
        test_b.schema
    )
    assert test_b.columns == DRIFT_METRIC_COLUMNS
    assert test_a.first()["valor"] - test_b.first()["valor"] == -15.0
    assert DRIFT_METRIC_DIRECTIONS["pct_preenchimento"] == "HIGHER_BETTER"
    assert {"interpretacao", "delta_absoluto"}.issubset(drift_metrics_df.columns)
    print("Testes sintéticos básicos aprovados.")
else:
    print("Testes sintéticos desabilitados.")

In [0]:
# ============================================================
# 17.14 CONTRATOS E SAÍDAS FINAIS
# ============================================================

# O QUE FAZ:
# Confere os DataFrames publicados pelo módulo.

# COMO FAZ:
# Valida presença de colunas e estado de execução.

# POR QUE É IMPORTANTE:
# Preserva a integração com os próximos módulos.

# PERGUNTA RESPONDIDA:
# "As saídas esperadas estão disponíveis?"


assert {"papel", "run_id", "dataset_id"}.issubset(drift_sources_df.columns)
assert {"status_comparabilidade", "metrica"}.issubset(drift_comparability_df.columns)
assert {"delta_absoluto", "delta_relativo_pct", "interpretacao"}.issubset(drift_metrics_df.columns)
assert {"atributo", "evento"}.issubset(drift_schema_df.columns)
assert {"status", "pct_comparabilidade", "schema_status"}.issubset(drift_summary_df.columns)
assert {"categoria", "classificacao", "quantidade"}.issubset(drift_coverage_df.columns)
print("Contratos do módulo 17 validados.")
if DRIFT_DISPLAY_RESULTS:
    display(drift_summary_df)
    if DRIFT_ENABLE:
        display(drift_coverage_df)
        display(drift_metrics_df.filter(F.col("interpretacao") != "ESTAVEL"))
        if drift_schema_status == "AVALIADO":
            display(drift_schema_df.filter(F.col("evento") != "SEM_ALTERACAO"))